In [6]:
import xarray as xr
import xesmf as xe
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
from pathlib import Path
import cartopy.crs as ccrs
import cartopy as cart
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors

In [7]:
# =========================
# SETTINGS
# =========================
Nreplete = 0.925
deglim = 0.5

# Target grid (1°)
ds_out = xr.Dataset({
    "lon": (["lon"], np.arange(0, 360, 1)),
    "lat": (["lat"], np.arange(-89.5, 90.5, 1))
})

# Load mask (convert your .mat to .nc beforehand)
mask = xr.open_dataset("data/woa_grid.nc")["M3d"].isel(depth=0)

display_labels = {
    'N':        1,
    'Fe':       2,
    'P':        3,
    'Si':       4,
    'Replete':  5,
    'SST':      6,
}

In [8]:

# =========================
# FUNCTION
# =========================
import xarray as xr
import xesmf as xe
import numpy as np

def process_dataset(ncfile, ds_out, M3d, regridder=None):
    """
    Load a POP2 dataset, regrid to 180x360, mask land, and compute phyto limitation.
    
    Parameters
    ----------
    ncfile : str
        Path to POP2 .nc file
    ds_out : xarray.Dataset
        Target grid (lon, lat)
    M3d : np.ndarray
        Land mask (shape ny,nx)
    regridder : xe.Regridder or None
        Pre-built regridder. If None, it will be created.
    
    Returns
    -------
    phyto_lim : np.ndarray
        3D array (ny,nx,3) with limitation codes
    regridder : xe.Regridder
        Regridder object (can reuse for multiple files)
    """
    ds = xr.open_dataset(ncfile)
    
    # Build regridder if not supplied
    if regridder is None:
        ds_in_grid = xr.Dataset({"lat": ds["TLAT"], "lon": ds["TLONG"]})
        regridder = xe.Regridder(ds_in_grid, ds_out, method="nearest_s2d", periodic=True, reuse_weights=False)
    
    def R(var):
        """Regrid and mask a variable at the top depth slice."""
        da = ds[var]

        # Determine which vertical dimension exists
        if 'z_t' in da.dims:
            zdim = 'z_t'
        elif 'z_t_150m' in da.dims:
            zdim = 'z_t_150m'
        else:
            zdim = None  # variable has no vertical dimension

        # Take the top slice if vertical dimension exists
        if zdim:
            v = da.isel({zdim: 0}).values  # top layer
        else:
            v = da.values  # no vertical dimension

        # Regrid
        v = regridder(v)

        # Mask inland/land
        v[v == 0] = np.nan
        v = v[0, :, :]  
        v[M3d.T[:,:] == 0] = np.nan  # transpose mask if needed

        return v
    
    # Small phyto
    spN  = R("sp_N_lim")
    spFe = R("sp_Fe_lim")
    spP  = R("sp_P_lim")
    spPar = R("sp_light_lim")
    spC  = R("spC")
    
    # Diatoms
    diatN  = R("diat_N_lim")
    diatFe = R("diat_Fe_lim")
    diatP  = R("diat_P_lim")
    diatSi = R("diat_SiO3_lim")
    diatPar = R("diat_light_lim")
    diatC  = R("diatC")
    
    # Diazotrophs
    diazFe = R("diaz_Fe_lim")
    diazP  = R("diaz_P_lim")
    diazPar = R("diaz_light_lim")
    diazC  = R("diazC")
    TEMP   = R("TEMP")
    
    # Initialize limitation array
    ny, nx = diatN.shape
    phyto_lim = np.full((ny, nx, 3), np.nan)
    
    # Constants for coloring/limitation codes
    Nreplete = 0.925
    deglim   = 0.5
    col = {
        "N":       1,
        "Fe":      2,
        "P":       3,
        "Si":      4,
        "Replete": 5,
        "SST":     6
    }
    
    # --- Diatoms limitation ---
    min_all = np.stack([diatN, diatFe, diatSi, diatP], axis=0)
    min_val = np.nanmin(min_all, axis=0)
    
    phyto_lim[:,:,0] = np.where(diatN  == min_val, col["N"],       phyto_lim[:,:,0])
    phyto_lim[:,:,0] = np.where(diatFe == min_val, col["Fe"],      phyto_lim[:,:,0])
    phyto_lim[:,:,0] = np.where(diatSi == min_val, col["Si"],      phyto_lim[:,:,0])
    phyto_lim[:,:,0] = np.where(diatP  == min_val, col["P"],       phyto_lim[:,:,0])
    phyto_lim[:,:,0] = np.where(min_val > Nreplete, col["Replete"], phyto_lim[:,:,0])
    
    # --- Small phyto limitation ---
    min_sp  = np.stack([spN, spFe, spP], axis=0)
    min_val = np.nanmin(min_sp, axis=0)
    
    phyto_lim[:,:,1] = np.where(spN   == min_val, col["N"],       phyto_lim[:,:,1])
    phyto_lim[:,:,1] = np.where(spFe  == min_val, col["Fe"],      phyto_lim[:,:,1])
    phyto_lim[:,:,1] = np.where(spP   == min_val, col["P"],       phyto_lim[:,:,1])
    phyto_lim[:,:,1] = np.where(min_val > Nreplete, col["Replete"], phyto_lim[:,:,1])
    
    # --- Diazotroph limitation ---
    phyto_lim[:,:,2] = np.where(diazFe <  diazP,   col["Fe"],      phyto_lim[:,:,2])
    phyto_lim[:,:,2] = np.where(diazP  <  diazFe,  col["P"],       phyto_lim[:,:,2])
    phyto_lim[:,:,2] = np.where(
        (diazFe >= Nreplete) & (diazP >= Nreplete), col["Replete"], phyto_lim[:,:,2]
    )
    phyto_lim[:,:,2] = np.where(TEMP < 15, col["SST"], phyto_lim[:,:,2])
    
    return phyto_lim, regridder



In [9]:
hex_colors = [
    '#FFFFFF',  # 0  — Unset
    '#D0B425',  # 1  — N
    '#8357BA',  # 2  — Fe
    '#5AAAD8',  # 3  — P
    '#34997F',  # 4  — Si
    '#E8853A',  # 5  — Replete
    '#888780',  # 6  — SST
]

cmap = ListedColormap(hex_colors, name='phyto_limitation', N=7)

# =========================
# PLOTTING
# =========================
def plot_map(data, title, filename, cmap):
    fig = plt.figure(figsize=(8, 4), dpi=300)
    ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=-180))
    ax.add_feature(cart.feature.LAND, facecolor='lightgray', zorder=0)
    ax.set_global()
    ax.coastlines(zorder=2)

    # Gridlines and labels
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.xlocator = ticker.FixedLocator([-180, -90, 0, 90, 180])
    gl.ylocator = ticker.FixedLocator([-90, -45, 0, 45, 90])
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    gl.xlabel_style = {'size': 9, 'color': 'black'}
    gl.ylabel_style = {'size': 9, 'color': 'black'}
    gl.xformatter = LongitudeFormatter(zero_direction_label=False, degree_symbol='°')
    gl.yformatter = LatitudeFormatter(degree_symbol='°')

    # --- KEY FIX: BoundaryNorm pins each integer to exactly one colour ---
    n_colors = len(hex_colors)                        # 11
    bounds = np.arange(-0.5, 7.5, 1)      # [-0.5, 0.5, 1.5, ..., 10.5]
    norm = BoundaryNorm(bounds, cmap.N)

    # Plot the data
    mesh = ax.pcolormesh(
        ds_out["lon"], ds_out["lat"], data,
        shading="auto",
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        zorder=1
    )

    # Discrete colorbar with one tick per code
#    cbar_labels = [''] * len(hex_colors)
#    for label, idx in display_labels.items():
#        cbar_labels[idx] = label
#    cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', fraction=0.03, pad=0.05)
#    cbar.set_ticks(np.arange(0, n_colors))
#    cbar.ax.set_yticklabels(cbar_labels, fontsize=7)
    ax.set_rasterization_zorder(1)

    #plt.title(title)
    plt.savefig(filename, dpi=600, bbox_inches='tight',
            transparent=False)
    plt.close(fig)


In [10]:


# =========================
# MAIN
# =========================
files = [
    "data/gdev1440.pop.h.0281_0300.nc",
    "data/gdev1420.pop.h.0281_0300.nc"
]

regridder = None

for f in files:
    name = Path(f).stem

    phyto_lim, regridder = process_dataset(f, ds_out, mask, regridder)

    plot_map(phyto_lim[:,:,0], "Diatom limitation", f"figures/{name}_diat_simplified.svg", cmap)
    plot_map(phyto_lim[:,:,1], "Small phyto limitation", f"figures/{name}_sp_simplified.svg", cmap)
    plot_map(phyto_lim[:,:,2], "Diazotroph limitation", f"figures/{name}_diaz_simplified.svg", cmap)

print("All datasets processed.")

/tmp/ipykernel_36603/1815602262.py:30: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(ncfile)
/tmp/ipykernel_36603/1815602262.py:105: RuntimeWarning: All-NaN slice encountered
  min_val = np.nanmin(min_all, axis=0)
/tmp/ipykernel_36603/1815602262.py:115: RuntimeWarning: All-NaN slice encountered
  min_val = np.nanmin(min_sp, axis=0)
/tmp/ipykernel_36603/1815602262.py:30: FutureWarning: In a future version, xarray will not decode the variable 'days_in

All datasets processed.
